In [2]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

file_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\UpdatedResumeDataSet.csv'
resumeDataSet = pd.read_csv(file_path, encoding='utf-8')

def cleanResume(resumeText):
    resumeText = re.sub('httpS+s*', ' ', resumeText)
    resumeText = re.sub('RT|cc', ' ', resumeText)
    resumeText = re.sub('#S+', '', resumeText)
    resumeText = re.sub('@S+', ' ', resumeText)
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[]^_`{|}~"""), ' ', resumeText)
    resumeText = re.sub(r'[^x00-x7f]', r' ', resumeText)
    resumeText = re.sub('s+', ' ', resumeText)
    return resumeText

resumeDataSet['cleaned_resume'] = resumeDataSet.Resume.apply(lambda x: cleanResume(x))

print(resumeDataSet[['Resume', 'cleaned_resume']].head())

le = LabelEncoder()
resumeDataSet['Category'] = le.fit_transform(resumeDataSet['Category'])

word_vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words='english', max_features=1500)
WordFeatures = word_vectorizer.fit_transform(resumeDataSet['cleaned_resume'].values)

preprocessed_file_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\PreprocessedResumeDataSet.csv'
resumeDataSet.to_csv(preprocessed_file_path, index=False, encoding='utf-8')
print(f'Preprocessed data saved to {preprocessed_file_path}')


                                              Resume  \
0  Skills * Programming Languages: Python (pandas...   
1  Education Details \r\nMay 2013 to May 2017 B.E...   
2  Areas of Interest Deep Learning, Control Syste...   
3  Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...   
4  Education Details \r\n MCA   YMCAUST,  Faridab...   

                                      cleaned_resume  
0  Skill    Programming Language   P thon  panda ...  
1  Education Detail    Ma  2013 to Ma  2017 B E  ...  
2  Area  of Intere t Deep Learning  Control S  te...  
3  Skill      R     P thon     SAP HANA     Table...  
4  Education Detail     MCA   YMCAUST   Faridabad...  
Preprocessed data saved to C:\Users\Jam Sahir Raza\Desktop\resume analyzer\PreprocessedResumeDataSet.csv


now we have trained our dataset upon LSTM model and it gives us the 62% accuracy 

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn import metrics

file_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\PreprocessedResumeDataSet.csv'
resumeDataSet = pd.read_csv(file_path, encoding='utf-8')

resumeDataSet.reset_index(drop=True, inplace=True)

max_features = 2000
tokenizer = Tokenizer(num_words=max_features, split=' ')
tokenizer.fit_on_texts(resumeDataSet['cleaned_resume'].values)
X = tokenizer.texts_to_sequences(resumeDataSet['cleaned_resume'].values)
X = pad_sequences(X, maxlen=500)

labelencoder = LabelEncoder()
integer_encoded = labelencoder.fit_transform(resumeDataSet['Category'])
y = to_categorical(integer_encoded)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = Sequential()
model.add(Embedding(max_features, 128, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.4))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(y.shape[1], activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

batch_size = 32
model.fit(X_train, y_train, epochs=5, batch_size=batch_size, verbose=2)

score, acc = model.evaluate(X_test, y_test, verbose=2, batch_size=batch_size)
print("Score: %.2f" % (score))
print("Validation Accuracy: %.2f" % (acc))

model.save('complex_resume_model.h5')
print("Model saved as complex_resume_model.h5")


c:\Users\Jam Sahir Raza\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
25/25 - 17s - 669ms/step - accuracy: 0.1014 - loss: 3.1878
Epoch 2/5
25/25 - 11s - 442ms/step - accuracy: 0.3160 - loss: 2.9896
Epoch 3/5
25/25 - 11s - 426ms/step - accuracy: 0.3433 - loss: 2.6663
Epoch 4/5
25/25 - 11s - 442ms/step - accuracy: 0.4317 - loss: 2.2722
Epoch 5/5
25/25 - 10s - 396ms/step - accuracy: 0.5670 - loss: 1.8690
7/7 - 1s - 174ms/step - accuracy: 0.6218 - loss: 1.6691


Score: 1.67
Validation Accuracy: 0.62
Model saved as complex_resume_model.h5


this code uses CNN MODEL to classify the resume texts etc 

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Conv1D, GlobalMaxPooling1D, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

file_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\PreprocessedResumeDataSet.csv'
data = pd.read_csv(file_path, encoding='utf-8')

max_features = 2000
maxlen = 500
embedding_dims = 50

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(data['cleaned_resume'])
sequences = tokenizer.texts_to_sequences(data['cleaned_resume'])
x_data = pad_sequences(sequences, maxlen=maxlen)

label_encoder = LabelEncoder()
y_data = label_encoder.fit_transform(data['Category'])
y_data = to_categorical(y_data)

x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=0)

model = Sequential()
model.add(Embedding(max_features, embedding_dims, input_length=maxlen))
model.add(Dropout(0.2))
model.add(Conv1D(256, 3, padding='valid', activation='relu', strides=1))
model.add(GlobalMaxPooling1D())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(len(label_encoder.classes_), activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(x_train, y_train, batch_size=32, epochs=10, validation_data=(x_test, y_test))

model.save('cnn_resume_classifier.h5')
print("Model saved as cnn_resume_classifier.h5")


Epoch 1/10


c:\Users\Jam Sahir Raza\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 139ms/step - accuracy: 0.0715 - loss: 3.2023 - val_accuracy: 0.0984 - val_loss: 3.1226
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 120ms/step - accuracy: 0.0830 - loss: 3.1169 - val_accuracy: 0.1140 - val_loss: 2.9983
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 127ms/step - accuracy: 0.1141 - loss: 2.9426 - val_accuracy: 0.2539 - val_loss: 2.7229
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 123ms/step - accuracy: 0.3013 - loss: 2.6707 - val_accuracy: 0.4352 - val_loss: 2.2931
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 148ms/step - accuracy: 0.4737 - loss: 2.1684 - val_accuracy: 0.5492 - val_loss: 1.7526
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 137ms/step - accuracy: 0.5987 - loss: 1.6171 - val_accuracy: 0.7150 - val_loss: 1.2602
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 117ms/step - accuracy: 0.7185 - loss: 1.1379 - val_accuracy: 0.8083 - val_loss: 0.8497
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 125ms/step - accuracy: 0.8535 - loss: 0.7151 - val_accuracy: 0.9016 - val_

Model saved as cnn_resume_classifier.h5


In this code we have used the Naive Bayes classifier with TF-IDF technique to classify resume texts 

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer
import re

file_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\PreprocessedResumeDataSet.csv'
resumeDataSet = pd.read_csv(file_path, encoding='utf-8')

resumeDataSet.reset_index(drop=True, inplace=True)

word_vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words='english', max_features=1500)
WordFeatures = word_vectorizer.fit_transform(resumeDataSet['cleaned_resume'].values)

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    WordFeatures, resumeDataSet['Category'], resumeDataSet.index, test_size=0.2, random_state=0
)

clf = MultinomialNB()
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)

print('Accuracy on training set: {:.2f}'.format(clf.score(X_train, y_train)))
print('Accuracy on test set: {:.2f}'.format(clf.score(X_test, y_test)))
print(metrics.classification_report(y_test, predictions))

def extract_resume_details(text):
    skills = re.findall(r"(Python|Java|JavaScript|SQL|React|C\+\+|C#|Salesforce|Machine Learning|AI)", text, flags=re.I)
    education = re.findall(r"(Bachelor|Master|PhD|B\.Sc|M\.Sc|B\.A|M\.A|University|College|School)", text, flags=re.I)
    experience = re.findall(r"(\d+\s+years?)\s+experience", text, flags=re.I)
    return {
        'Skills': ', '.join(sorted(set(skills))) if skills else 'No specific skills listed',
        'Education': ', '.join(sorted(set(education))) if education else 'No education details listed',
        'Experience': ', '.join(f"{x[0]} years" for x in set(experience)) if experience else 'No experience listed'
    }

predicted_data = pd.DataFrame({
    'Resume Text': resumeDataSet.loc[idx_test, 'cleaned_resume'],
    'Predicted Category': predictions,
    'Actual Category': y_test.array
}).reset_index(drop=True)

predicted_data['Detailed Insights'] = predicted_data['Resume Text'].apply(lambda x: extract_resume_details(x))

predicted_data.to_csv('Detailed_Resume_Insights.csv', index=False)
print("Detailed resume insights have been saved to 'Detailed_Resume_Insights.csv'")

print("Sample Resume Insights:")
print(predicted_data[['Resume Text', 'Detailed Insights']].head())


Accuracy on training set: 0.97
Accuracy on test set: 0.92
              precision    recall  f1-score   support

           0       1.00      0.67      0.80         3
           1       0.75      1.00      0.86         3
           2       1.00      0.80      0.89         5
           3       1.00      1.00      1.00         9
           4       1.00      0.83      0.91         6
           5       1.00      1.00      1.00         5
           6       1.00      0.78      0.88         9
           7       1.00      1.00      1.00         7
           8       1.00      0.91      0.95        11
           9       1.00      0.56      0.71         9
          10       1.00      1.00      1.00         8
          11       1.00      0.44      0.62         9
          12       1.00      1.00      1.00         5
          13       1.00      1.00      1.00         9
          14       1.00      1.00      1.00         7
          15       0.73      1.00      0.84        19
          16       1.00

In [6]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer

file_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\PreprocessedResumeDataSet.csv'
resumeDataSet = pd.read_csv(file_path, encoding='utf-8')

word_vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words='english', max_features=1500)
WordFeatures = word_vectorizer.fit_transform(resumeDataSet['cleaned_resume'].values)

X_train, X_test, y_train, y_test = train_test_split(
    WordFeatures, resumeDataSet['Category'].values, random_state=0, test_size=0.2
)

clf = MultinomialNB()
clf.fit(X_train, y_train)

prediction = clf.predict(X_test)

print('Accuracy of MultinomialNB on training set: {:.2f}'.format(clf.score(X_train, y_train)))
print('Accuracy of MultinomialNB on test set: {:.2f}'.format(clf.score(X_test, y_test)))
print(metrics.classification_report(y_test, prediction))

model_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\model.pkl'
vectorizer_path = r'C:\Users\Jam Sahir Raza\Desktop\resume analyzer\vectorizer.pkl'

with open(model_path, 'wb') as model_file:
    pickle.dump(clf, model_file)

with open(vectorizer_path, 'wb') as vectorizer_file:
    pickle.dump(word_vectorizer, vectorizer_file)

print(f"Model saved to {model_path}")
print(f"Vectorizer saved to {vectorizer_path}")


Accuracy of MultinomialNB on training set: 0.97
Accuracy of MultinomialNB on test set: 0.92
              precision    recall  f1-score   support

           0       1.00      0.67      0.80         3
           1       0.75      1.00      0.86         3
           2       1.00      0.80      0.89         5
           3       1.00      1.00      1.00         9
           4       1.00      0.83      0.91         6
           5       1.00      1.00      1.00         5
           6       1.00      0.78      0.88         9
           7       1.00      1.00      1.00         7
           8       1.00      0.91      0.95        11
           9       1.00      0.56      0.71         9
          10       1.00      1.00      1.00         8
          11       1.00      0.44      0.62         9
          12       1.00      1.00      1.00         5
          13       1.00      1.00      1.00         9
          14       1.00      1.00      1.00         7
          15       0.73      1.00      0.84